In [1]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    AgentEndpointConfig,
    FixedRatioVersionSelectionRule,
    VersionSelector,
)
from azure.identity import DefaultAzureCredential

In [2]:
PROJECT_ENDPOINT = "https://foundry-rag-nagh.services.ai.azure.com/api/projects/rag"

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, StructuredInputDefinition
from azure.identity import DefaultAzureCredential

# Format: "https://resource_name.ai.azure.com/api/projects/project_name"

# Create clients to call Foundry API
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=DefaultAzureCredential())
openai = project.get_openai_client()

# Create agent with handlebar templates in instructions
agent = project.agents.create_version(
    agent_name="structured-input-agent",
    definition=PromptAgentDefinition(
        model="gpt-4.1",
        instructions=(
            "You are a helpful assistant. "
            "The user's name is {{userName}} and their role is {{userRole}}. "
            "Greet them and confirm their details."
        ),
        structured_inputs={
            "userName": StructuredInputDefinition(
                description="The user's name", required=True, schema={"type": "string"},
            ),
            "userRole": StructuredInputDefinition(
                description="The user's role", required=True, schema={"type": "string"},
            ),
        },
    ),
)
print(f"Agent created: {agent.name}, version: {agent.version}")

# Create conversation and send request with runtime values
conversation = openai.conversations.create()
response = openai.responses.create(
    conversation=conversation.id,
    input="Hello! Can you confirm my details?",
    extra_body={
        "agent_reference": {"name": agent.name, "type": "agent_reference"},
        "structured_inputs": {"userName": "Alice Smith", "userRole": "Senior Developer"},
    },
)
print(response.output_text)

Agent created: structured-input-agent, version: 1
Hello, Alice Smith! You are listed as a Senior Developer. Please let me know if these details are correct or if there’s anything you’d like to update.
